# 05. 패션 추천 시스템 — 통합 파이프라인

| 단계 | 모델 | 역할 |
|------|------|------|
| 1 | 모델 A (ResNet18) | 이미지 → 스타일 24개 + 대분류 5개 분류 |
| 2 | 모델 B (KNN) | 키·몸무게+핏 선호도 → 사이즈 (XS~XL) 예측 |
| 3 | 필터링 | 스타일 상위 2개 + 예측 사이즈로 후보 압축 |
| 4 | 모델 C (Pandas) | 트렌드 점수로 최종 정렬 |

In [ ]:
import pickle
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import pandas as pd
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')

In [ ]:
# ── 클래스 정의 (학습된 모델과 동일) ──────────────────────
STYLE_CLASSES = [
    '기타', '레트로', '로맨틱', '리조트', '매니시',
    '모던', '밀리터리', '섹시', '소피스트케이티드', '스트리트',
    '스포티', '아방가르드', '오리엔탈', '웨스턴', '젠더리스',
    '컨트리', '클래식', '키치', '톰보이', '펑크',
    '페미닌', '프레피', '히피', '힙합',
]
CATEGORY_CLASSES = ['상의', '하의', '아우터', '원피스', '점프수트']
SIZE_CLASSES = ['XS', 'S', 'M', 'L', 'XL']
IMG_SIZE     = 224

EVAL_TRANSFORM = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# ── 모델 A 클래스 ─────────────────────────────────────────
class StyleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.style_head    = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(512, len(STYLE_CLASSES)))
        self.category_head = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(512, len(CATEGORY_CLASSES)))
    def forward(self, x):
        feat = self.backbone(x)
        return self.style_head(feat), self.category_head(feat)

In [ ]:
# ── 모델 로드 ────────────────────────────────────────────
model_a = StyleCNN()
model_a.load_state_dict(torch.load(r'C:\딥러닝데이터\model_a_cnn.pth', map_location=device, weights_only=True))
model_a.to(device).eval()

with open(r'C:\딥러닝데이터\model_b_knn.pkl', 'rb') as f:
    knn_data = pickle.load(f)
knn    = knn_data['knn']
scaler = knn_data['scaler']

print('모델 A (ResNet18) 로드 완료')
print('모델 B (KNN)      로드 완료')

In [ ]:
# ── 트렌드 점수 함수 (모델 C) ────────────────────────────
def rank_candidates(candidate_ids, top_k=5):
    products = pd.read_csv('data/상품목록.csv', encoding='utf-8-sig')
    sales    = pd.read_csv('data/판매데이터.csv', encoding='utf-8-sig')
    df = products.merge(sales, on='product_id')
    def minmax(col): return (col - col.min()) / (col.max() - col.min() + 1e-8)
    df['trend_score'] = 0.6*minmax(df['sales_7d']) + 0.3*minmax(df['sales_30d']) + 0.1*minmax(df['view_count'])
    return (
        df[df['product_id'].isin(candidate_ids)]
        .sort_values('trend_score', ascending=False)
        .head(top_k)
    )

In [ ]:
# ── 추천 함수 ────────────────────────────────────────────
def predict_size(height_cm, weight_kg, fit='regular'):
    x = scaler.transform([[height_cm, weight_kg]])
    proba = knn.predict_proba(x)[0]
    prob_dict = dict(zip(knn.classes_, proba))
    base = knn.predict(x)[0]
    idx  = SIZE_CLASSES.index(base)
    if fit == 'over' and idx < len(SIZE_CLASSES) - 1:
        ns = SIZE_CLASSES[idx + 1]
        if prob_dict.get(ns, 0) >= 0.3:
            return base, ns
    elif fit == 'slim' and idx > 0:
        ns = SIZE_CLASSES[idx - 1]
        if prob_dict.get(ns, 0) >= 0.3:
            return base, ns
    return base, base

def recommend(height_cm, weight_kg, fit='regular', img_path=None, top_k=5):
    # Step 1. 모델 A — 스타일 분류
    if img_path:
        image_tensor = EVAL_TRANSFORM(Image.open(img_path).convert('RGB')).unsqueeze(0)
    else:
        image_tensor = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

    with torch.no_grad():
        out_s, out_c = model_a(image_tensor.to(device))
        style_probs = torch.softmax(out_s, dim=1).squeeze().cpu().numpy()
        cat_probs   = torch.softmax(out_c, dim=1).squeeze().cpu().numpy()

    top_styles = [STYLE_CLASSES[i] for i in style_probs.argsort()[::-1][:2]]
    top_cat    = CATEGORY_CLASSES[cat_probs.argmax()]

    print('\n[모델 A] 스타일 분석 (상위 5개)')
    for i in style_probs.argsort()[::-1][:5]:
        print(f'  {STYLE_CLASSES[i]:16s} {"█"*int(style_probs[i]*20):<20s} {style_probs[i]:.1%}')
    print(f'  대분류 예측: {top_cat} ({cat_probs.max():.1%})')

    # Step 2. 모델 B — 사이즈 예측 + 핏 보정
    base_size, rec_size = predict_size(height_cm, weight_kg, fit)
    fit_label = {'slim': '슬림핏', 'regular': '레귤러핏', 'over': '오버핏'}[fit]
    if base_size == rec_size:
        print(f'\n[모델 B] 사이즈 예측: {rec_size}  ({fit_label} / 키 {height_cm}cm / {weight_kg}kg)')
    else:
        print(f'\n[모델 B] 사이즈 예측: {base_size} → {rec_size}  ({fit_label} 보정 / 키 {height_cm}cm / {weight_kg}kg)')

    # Step 3. 후보 필터링
    products = pd.read_csv('data/상품목록.csv', encoding='utf-8-sig')
    mask     = products['style'].isin(top_styles) & (products['size'] == rec_size)
    candidates = products[mask]['product_id'].tolist()
    if not candidates:
        candidates = products[products['style'].isin(top_styles)]['product_id'].tolist()
    print(f'\n[필터링] 스타일: {top_styles} / 사이즈: {rec_size} → 후보 {len(candidates)}개')

    # Step 4. 모델 C — 트렌드 정렬
    ranked = rank_candidates(candidates, top_k)
    return ranked

In [ ]:
# ── 실행 예시 ────────────────────────────────────────────
# fit 옵션: 'slim' / 'regular' / 'over'
# 실제 이미지 사용 시 img_path 지정

result = recommend(height_cm=168, weight_kg=60, fit='regular', top_k=5)

print(f'\n{"="*58}')
print(f'  최종 추천 TOP 5  (트렌드 점수 높은 순)')
print(f'{"="*58}')
result[['product_name', 'style', 'size', 'price', 'sales_7d', 'trend_score']]